# Tower of Hanoi

An interactive Tower of Hanoi game with an optimal auto-solver.

**How to use:** run all cells (Run All), then use the game in section 5. Click a tower to pick up its top disk, then click the destination tower. Click **Solve** to watch the optimal solution play out.

Setup instructions are in the README.

In [ ]:
import time

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from matplotlib.patches import Rectangle

PEG_NAMES = "ABC"

## 1. Simulator
Pegs are indexed 0, 1, 2 (A, B, C). Disks are numbered 1 (smallest) to n. All rule enforcement lives here, independent of the solver and the UI.

In [ ]:
class Hanoi:
    def __init__(self, n):
        if n < 1:
            raise ValueError("n must be >= 1")
        self.n = n
        self.reset()

    def reset(self):
        self.pegs = [list(range(self.n, 0, -1)), [], []]  # bottom -> top
        self.history = []

    def move(self, src, dst):
        if src == dst or src not in (0, 1, 2) or dst not in (0, 1, 2):
            raise ValueError(f"Invalid pegs: {src} -> {dst}")
        if not self.pegs[src]:
            raise ValueError(f"Tower {PEG_NAMES[src]} is empty")
        disk = self.pegs[src][-1]
        if self.pegs[dst] and self.pegs[dst][-1] < disk:
            raise ValueError(f"Can't place disk {disk} on smaller disk {self.pegs[dst][-1]}")
        self.pegs[dst].append(self.pegs[src].pop())
        self.history.append((src, dst))

    def is_solved(self):
        return len(self.pegs[2]) == self.n

    def draw(self, title=""):
        fig, ax = plt.subplots(figsize=(8, 3))
        for p in range(3):
            cx = p * 4 + 2
            ax.add_patch(Rectangle((cx - 0.05, 0), 0.1, self.n + 1, color="black"))
            for level, disk in enumerate(self.pegs[p]):
                w = 0.6 + 3.2 * disk / self.n
                ax.add_patch(Rectangle((cx - w / 2, level), w, 0.9,
                                       color=plt.cm.viridis(disk / self.n), ec="black"))
            ax.text(cx, -0.8, PEG_NAMES[p], ha="center", fontsize=12)
        ax.set_xlim(0, 12)
        ax.set_ylim(-1.2, self.n + 1.5)
        ax.axis("off")
        ax.set_title(title)
        plt.show()

## 2. Solver
The minimum number of moves for n disks is 2^n - 1. The recursive generator is the solver used by the game. The iterative version is an independent implementation used only to cross-check it.

In [ ]:
def solve_recursive(n, src=0, aux=1, dst=2):
    """Yield (src, dst) moves: n-1 disks to aux, disk n to dst, n-1 disks onto it."""
    if n == 0:
        return
    yield from solve_recursive(n - 1, src, dst, aux)
    yield (src, dst)
    yield from solve_recursive(n - 1, aux, src, dst)


def solve_iterative(n):
    """Cross-check solver. Alternates: move the smallest disk one step around a
    fixed cycle, then make the only legal move not involving it."""
    h = Hanoi(n)
    order = [0, 2, 1] if n % 2 else [0, 1, 2]  # cycle direction depends on parity
    small = 0
    while not h.is_solved():
        nxt = order[(order.index(small) + 1) % 3]
        h.move(small, nxt)
        small = nxt
        if h.is_solved():
            break
        a, b = [p for p in range(3) if p != small]
        if not h.pegs[a]:
            src, dst = b, a
        elif not h.pegs[b]:
            src, dst = a, b
        elif h.pegs[a][-1] < h.pegs[b][-1]:
            src, dst = a, b
        else:
            src, dst = b, a
        h.move(src, dst)
    return h.history

## 3. Verification
Replay both solvers through the simulator. Each must finish solved in exactly 2^n - 1 moves, and the two must produce identical sequences (the optimal solution is unique).

In [ ]:
def verify(moves, n):
    h = Hanoi(n)
    for s, d in moves:
        h.move(s, d)
    assert h.is_solved(), f"n={n}: not solved"
    assert len(moves) == 2**n - 1, f"n={n}: {len(moves)} moves, expected {2**n - 1}"


for n in range(1, 13):
    rec = list(solve_recursive(n))
    it = solve_iterative(n)
    verify(rec, n)
    verify(it, n)
    assert rec == it, f"solvers disagree at n={n}"
print("All checks passed for n = 1..12")

## 4. Game UI
A thin widget layer over the simulator. The Solve button resets the game and animates the optimal solution from the start.

In [ ]:
class HanoiApp:
    SOLVE_DELAY = 0.3  # seconds per move during Solve
    MAX_DISKS = 7      # 127 moves; larger values make the animation tedious

    def __init__(self, n=3):
        self.game = Hanoi(n)
        self.source = None   # tower currently selected as the move source
        self.busy = False    # True while Solve is animating

        self.out = widgets.Output()
        self.status = widgets.Label()
        self.disks = widgets.Dropdown(
            options=list(range(1, self.MAX_DISKS + 1)), value=n,
            description="Disks:", layout=widgets.Layout(width="150px"))
        self.towers = [widgets.Button(description=f"Tower {name}",
                                      layout=widgets.Layout(width="140px"))
                       for name in PEG_NAMES]
        self.reset_btn = widgets.Button(description="Reset", button_style="warning")
        self.solve_btn = widgets.Button(description="Solve", button_style="success")

        for i, b in enumerate(self.towers):
            b.on_click(lambda _, i=i: self._on_tower(i))
        self.reset_btn.on_click(lambda _: self._on_reset())
        self.solve_btn.on_click(lambda _: self._on_solve())
        self.disks.observe(self._on_disks, names="value")

        self.ui = widgets.VBox([
            widgets.HBox([self.disks, self.reset_btn, self.solve_btn]),
            self.out,
            widgets.HBox(self.towers, layout=widgets.Layout(
                justify_content="space-around", width="640px")),
            self.status,
        ])

    def show(self):
        display(self.ui)
        self._refresh()

    def _refresh(self, msg=""):
        locked = self.busy or self.game.is_solved()
        for i, b in enumerate(self.towers):
            b.disabled = locked
            b.button_style = "info" if self.source == i else ""
        self.disks.disabled = self.reset_btn.disabled = self.solve_btn.disabled = self.busy
        with self.out:
            clear_output(wait=True)
            self.game.draw(f"Moves: {len(self.game.history)}  (optimal: {2**self.game.n - 1})")
        self.status.value = msg

    def _on_tower(self, i):
        g = self.game
        if self.source is None:
            if not g.pegs[i]:
                return self._refresh(f"Tower {PEG_NAMES[i]} is empty")
            self.source = i
            return self._refresh(f"From {PEG_NAMES[i]}: pick a destination")
        src, self.source = self.source, None
        if src == i:
            return self._refresh("Cancelled")
        try:
            g.move(src, i)
        except ValueError as e:
            return self._refresh(f"Invalid: {e}")
        if g.is_solved():
            best = 2**g.n - 1
            note = "optimal!" if len(g.history) == best else f"optimal is {best}"
            self._refresh(f"Solved in {len(g.history)} moves ({note})")
        else:
            self._refresh()

    def _on_reset(self):
        self.game.reset()
        self.source = None
        self._refresh()

    def _on_disks(self, change):
        self.game = Hanoi(change.new)
        self.source = None
        self._refresh()

    def _on_solve(self):
        self.game.reset()
        self.source = None
        self.busy = True
        try:
            self._refresh("Solving...")
            for src, dst in solve_recursive(self.game.n):
                time.sleep(self.SOLVE_DELAY)
                self.game.move(src, dst)
                self._refresh("Solving...")
        finally:
            self.busy = False
        self._refresh(f"Solved in {len(self.game.history)} moves (optimal)")

## 5. Play

In [ ]:
app = HanoiApp(3)
app.show()